# "THE PRICE IS RIGHT"
with 200 datapoints and 50 validation set and 50 test data
<p>Final Graph to show statistic of what was correct and if wrong by how much</p>

In [ ]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [ ]:
# environment

LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
dataset = "ed-donner/items_lite"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
openai = OpenAI()

In [ ]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)
# Going with 200 examples and 1 epoch


fine_tune_train = train[:200]
fine_tune_validation = val[:50]

In [ ]:
len(fine_tune_train)

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [ ]:
messages_for(fine_tune_train[0])

In [ ]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:
print(make_jsonl(train[:3]))

In [ ]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [ ]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [ ]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:
# skip this step if you already have a finetuned model and do not want a new one
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
train_file

In [ ]:
# skip this step if you already have a finetuned model and do not want a new one
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
validation_file

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [ ]:
# skip this step if you already have a finetuned model and do not want a new one
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)

In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [ ]:
job_id

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [ ]:
#use this to get the finetuned model or retrieve it from https://platform.openai.com/finetune
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [ ]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [ ]:
# Try this out

test_messages_for(test[8])

In [ ]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [ ]:
print(test[8].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

In [ ]:
# Evaluation: random sample from test with color-coded results
import random
from pricer.evaluator import Tester

# User-defined sample size
eval_sample_size = 50  # change to evaluate more or fewer items

sample = random.sample(list(test), min(eval_sample_size, len(test)))

# Margins: correct = within 5% or $5; small = within 20% or $40; else big
def margin_kind(guess, truth):
    if truth == 0:
        return "exact" if guess == 0 else "big"
    err = abs(guess - truth)
    pct = err / truth
    if err < 5 or pct < 0.05:
        return "correct"
    if err < 40 or pct < 0.20:
        return "small"
    return "big"

# ANSI colors: below = blue/cyan, above = red/yellow, correct = green
GREEN, CYAN, BLUE, YELLOW, RED, RESET = "\033[92m", "\033[96m", "\033[94m", "\033[93m", "\033[91m", "\033[0m"

def style_and_emoji(guess, truth):
    kind = margin_kind(guess, truth)
    below = guess < truth
    if kind == "correct":
        return GREEN, "✓"
    if below:
        return (CYAN if kind == "small" else BLUE), "↓"
    return (YELLOW if kind == "small" else RED), "↑"

eval_guesses, eval_truths, eval_titles = [], [], []
print(f"Evaluating {len(sample)} random test items (sample size = {eval_sample_size})...\n")
for i, item in enumerate(sample):
    raw = gpt_4__1_nano_fine_tuned(item)
    guess = Tester.post_process(raw)
    truth = item.price
    eval_guesses.append(guess)
    eval_truths.append(truth)
    eval_titles.append(item.title[:50] + "..." if len(item.title) > 50 else item.title)
    color, emoji = style_and_emoji(guess, truth)
    title = eval_titles[-1]
    diff = guess - truth
    print(f"{color}{emoji} {RESET} ${guess:,.2f} (actual ${truth:,.2f}, {diff:+,.2f})  {title}")
print(f"\n{RESET}Legend: {GREEN}✓ correct{RESET} | {CYAN}↓ below (small){RESET} | {BLUE}↓ below (big){RESET} | {YELLOW}↑ above (small){RESET} | {RED}↑ above (big){RESET}")

In [ ]:
# Summary graph of evaluation results
import matplotlib.pyplot as plt
import numpy as np

# Reuse margin categories for point colors
def eval_category(guess, truth):
    if truth == 0:
        return "correct" if guess == 0 else "above_big"
    err = abs(guess - truth)
    pct = err / truth
    below = guess < truth
    if err < 5 or pct < 0.05:
        return "correct"
    if err < 40 or pct < 0.20:
        return "below_small" if below else "above_small"
    return "below_big" if below else "above_big"

categories = [eval_category(g, t) for g, t in zip(eval_guesses, eval_truths)]
colors = {"correct": "green", "below_small": "cyan", "below_big": "blue", "above_small": "gold", "above_big": "red"}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Actual vs predicted scatter
ax1 = axes[0]
for cat in colors:
    mask = [c == cat for c in categories]
    if not any(mask):
        continue
    x = [eval_truths[i] for i in range(len(mask)) if mask[i]]
    y = [eval_guesses[i] for i in range(len(mask)) if mask[i]]
    ax1.scatter(x, y, c=colors[cat], label=cat.replace("_", " "), alpha=0.8, s=60)
max_val = max(max(eval_truths), max(eval_guesses)) * 1.05
ax1.plot([0, max_val], [0, max_val], "k--", alpha=0.6, label="perfect")
ax1.set_xlabel("Actual price ($)")
ax1.set_ylabel("Predicted price ($)")
ax1.set_title("Predicted vs actual price")
ax1.legend(loc="upper left", fontsize=8)
ax1.set_xlim(0, max_val)
ax1.set_ylim(0, max_val)
ax1.grid(True, alpha=0.3)

# 2. Bar chart: count per category
ax2 = axes[1]
counts = {cat: categories.count(cat) for cat in colors}
names = list(counts.keys())
vals = [counts[n] for n in names]
bars = ax2.bar([n.replace("_", "\n") for n in names], vals, color=[colors[n] for n in names], edgecolor="gray")
ax2.set_ylabel("Count")
ax2.set_title("Results by category")
for b, v in zip(bars, vals):
    ax2.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.2, str(v), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

# Print short summary
n_correct = categories.count("correct")
mae = np.mean([abs(g - t) for g, t in zip(eval_guesses, eval_truths)])
print(f"Summary: {n_correct}/{len(eval_guesses)} correct ({100*n_correct/len(eval_guesses):.0f}%)  |  Mean absolute error: ${mae:.2f}")